# 02 · 链（Chains）与 LCEL

当「提示词 → 模型 → 解析」需要串联或组合时，就用 **链（Chain）**。现代 LangChain 推荐用 **LCEL（LangChain Expression Language）** 以管道符 `|` 组合组件。

运行前请确保：
- 已激活 `.venv` 虚拟环境
- 已配置 `.env` 中的 `OPENAI_API_KEY`

> 详细概念讲解见 `docs/02-chains.md`。


## 0. 环境检查


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
if not os.getenv('OPENAI_API_KEY'):
    raise SystemExit('✗ 未找到 OPENAI_API_KEY，请复制 .env.example 为 .env 并填写。')
print('✓ 环境就绪，API Key 已加载')


## 1. 最基础的链：提示词 | 模型 | 解析

用 `|` 把组件串起来，整条链本身也是一个可调用对象。


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model=os.getenv('OPENAI_MODEL', 'gpt-4o-mini'))
parser = StrOutputParser()

prompt = ChatPromptTemplate.from_messages([
    ('system', '你是一个翻译助手，只输出翻译结果，不要解释。'),
    ('human', '把下面内容翻译成{target_language}：\n{text}'),
])
translate_chain = prompt | llm | parser
print(translate_chain.invoke({'target_language': '英语', 'text': '你好，世界'}))


## 2. RunnableParallel：同一输入并行走多个分支

字典语法 `{...}` 会并行执行多个分支并合并结果。


In [ ]:
from langchain_core.runnables import RunnableParallel

summary_prompt = ChatPromptTemplate.from_template('用一句话总结：{text}')
keywords_prompt = ChatPromptTemplate.from_template('提取3个关键词（逗号分隔）：{text}')
parallel_chain = RunnableParallel({
    'summary': summary_prompt | llm | parser,
    'keywords': keywords_prompt | llm | parser,
})
result = parallel_chain.invoke({
    'text': 'LangChain 是一个用于构建大语言模型应用的框架，它把模型、提示词、记忆、检索等能力组合成链。'
})
print(result)


## 3. RunnablePassthrough：保留原始输入

`RunnablePassthrough` 原样透传输入，常用于把原始字段一并传给后续步骤。


In [ ]:
from langchain_core.runnables import RunnablePassthrough

echo_chain = {
    'answer': prompt | llm | parser,
    'original_text': RunnablePassthrough(),
} | (lambda x: f"问题文本：{x['original_text']['text']}\n翻译结果：{x['answer']}")
print(echo_chain.invoke({'target_language': '英语', 'text': '今天天气真好'}))


## 4. 流式输出

`.stream()` 逐块返回结果，适合聊天场景逐字显示。


In [ ]:
for chunk in translate_chain.stream({'target_language': '日语', 'text': '我爱学习编程'}):
    print(chunk, end='', flush=True)
print()


## 5. 下一步

- 多轮记忆：运行 `examples/03_memory.py`，参见 `docs/03-memory.md`

**常见坑**：链中各环节的「输入/输出字段名」要对得上，字段对不上是初学者最常见错误；LCEL 链默认不保存状态，需要多轮上下文请看下一章 Memory。
